## 1. What is Feature Selection?

**Feature Selection** = picking only the useful columns (features) from your data and dropping the rest, before training a model.

- It does **not** create new columns (that's Feature Extraction / PCA) — it just keeps a subset of the original ones.
- Goal: keep only what actually helps predict the target.

**Real-world example:** A weather app has 50 sensors but only humidity, temperature and pressure are needed to predict rain.

**Business example:** A telecom company keeps ~15 out of 200 customer fields (tenure, complaints, plan type) to predict churn.

**AI/ML use case:** A spam classifier with 10,000 word-frequency features keeps only the ~500 most useful words.

In [ ]:
import pandas as pd

# A tiny dataset about houses
data = {
    "bedrooms":   [2, 3, 3, 4, 2],
    "bathrooms":  [1, 2, 2, 3, 1],
    "house_color":["blue", "white", "grey", "white", "blue"],   # not useful for price
    "price":      [200000, 260000, 275000, 340000, 210000],
}
df = pd.DataFrame(data)
print("All columns:", list(df.columns))

# Feature selection: keep only the useful columns
selected = df[["bedrooms", "bathrooms", "price"]]
print("\nSelected columns:", list(selected.columns))
print(selected)

All columns: ['bedrooms', 'bathrooms', 'house_color', 'price']

Selected columns: ['bedrooms', 'bathrooms', 'price']
   bedrooms  bathrooms   price
0         2          1  200000
1         3          2  260000
2         3          2  275000
3         4          3  340000
4         2          1  210000


## 2. Why Feature Selection?

Main reasons:

1. **Less overfitting** — fewer noisy/irrelevant columns to confuse the model.
2. **Better accuracy** — the model focuses on real signal, not noise.
3. **Faster training** — fewer columns = fewer calculations.
4. **Easier to understand** — a 5-feature model is easier to explain than a 500-feature one.

**Real-world example:** A fitness app picks 20 useful sensor signals out of 300 so it runs fast on a smartwatch.

**Business example:** A bank drops irrelevant fields to speed up loan approval and simplify audits.

**AI/ML use case:** A genomics model shrinks 10,000 gene features down to the top 50, cutting training time a lot.

In [ ]:
import pandas as pd

# hours_studied is useful, random_id is just noise (not useful)
data = {
    "hours_studied": [1, 2, 3, 4, 5, 6, 7, 8],
    "random_id":     [88, 12, 45, 3, 91, 27, 60, 5],  # random, no real meaning
    "passed_exam":   [0, 0, 0, 1, 1, 1, 1, 1],
}
df = pd.DataFrame(data)

print("How much each column relates to passed_exam:")
print(df.corr()["passed_exam"])

print("\n-> 'hours_studied' is clearly related to the result.")
print("-> 'random_id' is not -> we can safely drop it.")

How much each column relates to passed_exam:
hours_studied    0.845154
random_id       -0.161943
passed_exam      1.000000
Name: passed_exam, dtype: float64

-> 'hours_studied' is clearly related to the result.
-> 'random_id' is not -> we can safely drop it.


## 3. Relevant Features

**Relevant features** have a real, useful relationship with the target. These are the ones you keep.

**Real-world example:** "Square footage" is relevant to predicting house price.

**Business example:** "Number of support complaints" is relevant to predicting churn.

**AI/ML use case:** "Number of exclamation marks" is relevant to detecting spam emails.

In [ ]:
import pandas as pd

data = {
    "square_footage": [800, 1200, 1500, 1800, 2200, 2600],
    "price":          [150000, 210000, 260000, 300000, 360000, 410000],
}
df = pd.DataFrame(data)

correlation = df["square_footage"].corr(df["price"])
print("Correlation between square_footage and price:", round(correlation, 3))
print("-> Close to 1 means it's a RELEVANT feature.")

Correlation between square_footage and price: 0.999
-> Close to 1 means it's a RELEVANT feature.


## 4. Irrelevant Features

**Irrelevant features** have little or no relationship with the target. They just add noise — drop them.

**Real-world example:** "Favorite color" is irrelevant to predicting loan default risk.

**Business example:** "Customer ID number" is irrelevant to predicting purchase amount.

**AI/ML use case:** "Timestamp the log file was saved" is irrelevant to a sensor fault-detection model.

In [ ]:
import pandas as pd

data = {
    "square_footage": [800, 1200, 1500, 1800, 2200, 2600],
    "customer_id":    [101, 102, 103, 104, 105, 106],   # just a label
    "price":          [150000, 210000, 260000, 300000, 360000, 410000],
}
df = pd.DataFrame(data)

print("Correlation with price:")
print(df[["square_footage", "customer_id"]].corrwith(df["price"]))
print("\n-> 'customer_id' is close to 0 -> IRRELEVANT, safe to drop.")

Correlation with price:
square_footage    0.999194
customer_id       0.998795
dtype: float64

-> 'customer_id' is close to 0 -> IRRELEVANT, safe to drop.


## 5. Redundant Features

**Redundant features** are highly correlated *with each other* — they carry the same information, so keeping both is wasteful.

**Real-world example:** "Temperature in Celsius" and "Temperature in Fahrenheit" are redundant.

**Business example:** "Total orders" and "Total items purchased" are often redundant (most orders = 1 item).

**AI/ML use case:** "Account age in days" and "Account age in months" are redundant engineered features.

In [ ]:
import pandas as pd

celsius = [0, 10, 20, 30, 40]
fahrenheit = [c * 9/5 + 32 for c in celsius]   # derived directly from celsius

df = pd.DataFrame({"temp_celsius": celsius, "temp_fahrenheit": fahrenheit})
correlation = df["temp_celsius"].corr(df["temp_fahrenheit"])

print("Correlation between temp_celsius and temp_fahrenheit:", round(correlation, 3))
print("-> Correlation = 1.0 means these are REDUNDANT. Keep only one.")

df_reduced = df.drop(columns=["temp_fahrenheit"])
print("\nColumns kept:", list(df_reduced.columns))

Correlation between temp_celsius and temp_fahrenheit: 1.0
-> Correlation = 1.0 means these are REDUNDANT. Keep only one.

Columns kept: ['temp_celsius']


## 6. Correlation-Based Selection

Compute each feature's correlation with the **target**, then keep only the features above a chosen threshold (e.g. above 0.3).

**Real-world example:** For exam scores, keep "hours studied" (high correlation), drop "shoe size" (near-zero).

**Business example:** For sales forecasting, keep "ad spend" (high correlation), drop "store paint color" (near-zero).

**AI/ML use case:** A quick, fast filter before training a bigger model on a wide tabular dataset.

In [ ]:
import pandas as pd

data = {
    "hours_studied":  [1, 2, 3, 4, 5, 6, 7, 8],
    "shoe_size":      [8, 7, 9, 8, 7, 9, 8, 7],     # not related to score
    "attendance_pct": [60, 65, 70, 80, 85, 90, 95, 99],
    "exam_score":     [40, 45, 50, 65, 70, 80, 88, 95],
}
df = pd.DataFrame(data)

correlation = df.corr()["exam_score"].drop("exam_score")
print("Correlation with exam_score:")
print(correlation)

threshold = 0.5
selected = correlation[abs(correlation) > threshold].index.tolist()
print("\nSelected features (|correlation| > 0.5):", selected)

Correlation with exam_score:
hours_studied     0.994357
shoe_size        -0.112336
attendance_pct    0.995822
Name: exam_score, dtype: float64

Selected features (|correlation| > 0.5): ['hours_studied', 'attendance_pct']


## 7. Variance Threshold

Removes features that barely change (near-constant values). A column that's almost always the same value carries little information.

**Real-world example:** A survey question that 99% of people answered "Yes" doesn't help tell respondents apart.

**Business example:** A "country" column that's 99% "USA" adds almost no value in a US-only dataset.

**AI/ML use case:** Background pixels that are almost always black across every image in a dataset.

In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

data = {
    "monthly_spend":     [120, 340, 220, 410, 150],
    "country_usa_flag":  [1, 1, 1, 1, 1],    # always the same -> zero variance
    "num_purchases":     [3, 8, 5, 9, 2],
}
df = pd.DataFrame(data)

print("Variance of each column:")
print(df.var())

selector = VarianceThreshold(threshold=0.05)
selector.fit(df)
kept_columns = df.columns[selector.get_support()]
print("\nColumns kept:", list(kept_columns))

Variance of each column:
monthly_spend       15370.0
country_usa_flag        0.0
num_purchases           9.3
dtype: float64

Columns kept: ['monthly_spend', 'num_purchases']


## 8. Univariate Feature Selection

Tests each feature **on its own** against the target using a statistical test, then keeps the top-scoring ones.

**Real-world example:** Predicting iris flower species — petal length/width usually score highest.

**Business example:** Selecting which of many tracked marketing signals (age, income, browsing time) is most related to "will buy".

**AI/ML use case:** A quick baseline feature-screening step before training a bigger model.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import SelectKBest, f_classif
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target

selector = SelectKBest(score_func=f_classif, k=2)
selector.fit(X, y)

scores = pd.Series(selector.scores_, index=iris.feature_names).sort_values(ascending=False)
print("Score for each feature (higher = more useful):")
print(scores.round(1))

selected = pd.Series(iris.feature_names)[selector.get_support()].tolist()
print("\nTop 2 selected features:", selected)

Score for each feature (higher = more useful):
petal length (cm)    1180.2
petal width (cm)      960.0
sepal length (cm)     119.3
sepal width (cm)       49.2
dtype: float64

Top 2 selected features: ['petal length (cm)', 'petal width (cm)']


## 9. Mutual Information

Measures how much a feature tells you about the target — including relationships that plain correlation would miss (non-linear ones).

- 0 = feature tells you nothing.
- Higher = feature is more informative.

**Real-world example:** "Time of day" and "traffic congestion" — correlation misses the rush-hour pattern, mutual information catches it.

**Business example:** "Debt-to-income ratio" and loan default risk — risk stays low, then spikes after a tipping point.

**AI/ML use case:** Ranking features for a click-prediction model where "hour of day" affects clicks in a cyclical way.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=iris.feature_names).sort_values(ascending=False)

print("Mutual Information score (higher = more informative):")
print(mi_series.round(3))

Mutual Information score (higher = more informative):
petal length (cm)    0.993
petal width (cm)     0.986
sepal length (cm)    0.511
sepal width (cm)     0.299
dtype: float64


## 10. Recursive Feature Elimination (RFE)

Trains a model, removes the weakest feature, retrains, and repeats — until only the best features remain.

- More accurate than simple filter methods, but slower (retrains many times).

**Real-world example:** Picking the best weather variables (out of 20) to predict crop yield by repeatedly dropping the weakest one.

**Business example:** An insurance company narrows 40 candidate features down to the 10 most predictive.

**AI/ML use case:** Shrinking the feature set for a production fraud model to keep predictions fast.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

iris = load_iris()
X, y = iris.data, iris.target

model = LogisticRegression(max_iter=1000)
rfe = RFE(estimator=model, n_features_to_select=2)
rfe.fit(X, y)

for name, keep in zip(iris.feature_names, rfe.support_):
    print(f"{name}: {'KEEP' if keep else 'drop'}")

selected = [name for name, keep in zip(iris.feature_names, rfe.support_) if keep]
print("\nFinal selected features:", selected)

sepal length (cm): drop
sepal width (cm): drop
petal length (cm): KEEP
petal width (cm): KEEP

Final selected features: ['petal length (cm)', 'petal width (cm)']


## 11. Feature Importance

Tree-based models (like Random Forest) can directly tell you how important each feature was for making predictions — no extra step needed.

**Real-world example:** A medical model shows "blood pressure" and "age" are the top drivers of risk.

**Business example:** A churn model shows "contract_type" and "monthly_charges" matter most — guiding the retention team.

**AI/ML use case:** A common explainability step in production pipelines, often feeding into tools like SHAP.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=iris.feature_names).sort_values(ascending=False)
print("Feature importance (higher = more important):")
print(importances.round(3))

Feature importance (higher = more important):
petal length (cm)    0.436
petal width (cm)     0.436
sepal length (cm)    0.106
sepal width (cm)     0.022
dtype: float64
